# Chapter 3 — Candidate Generation and Selection

**Book alignment:** current Chapter 3 · internal demo `Stage 02`

The shared demo package calls this **Stage 02** internally. The notebook number follows the book chapter number; the internal stage number is one lower.

**Question this notebook isolates:** Can generation and selection be measured separately when several complete proposals compete?


## Hypothesis

A candidate set can contain a successful complete proposal while a weak evaluator selects a failing one. A stronger evaluator should close the selection gap **without changing generation**.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "demo" / "agents-from-first-principles").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing demo/agents-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DEMO_ROOT = REPO_ROOT / "demo" / "agents-from-first-principles"
sys.path.insert(0, str(DEMO_ROOT))

from first_principles_agent.candidates import (
    Candidate,
    CandidateSelector,
    CandidateSet,
    FixedCandidateGenerator,
    FunctionCandidateEvaluator,
    measure_selection,
)

## Controlled experiment


In [ ]:
candidate_set = CandidateSet.from_iterable(
    [
        Candidate("A", "edit_file", "delimiter=','", "keeps existing comma behavior"),
        Candidate(
            "B", "edit_file", "delimiter='|'", "implements pipe-delimited records"
        ),
        Candidate(
            "C", "edit_file", "delimiter=None", "tries autodetection but violates task"
        ),
    ]
)

generator = FixedCandidateGenerator(candidate_set)
generated = generator.generate()


def true_task_success(candidate: Candidate) -> bool:
    return "delimiter='|'" in candidate.patch


weak_evaluator = FunctionCandidateEvaluator(
    lambda candidate: 1.0 if "comma" in candidate.rationale else 0.2
)
strong_evaluator = FunctionCandidateEvaluator(
    lambda candidate: 1.0 if true_task_success(candidate) else 0.0
)

weak_selected = CandidateSelector(weak_evaluator).select_complete(generated)
strong_selected = CandidateSelector(strong_evaluator).select_complete(generated)

weak = measure_selection(generated, weak_selected, true_task_success).as_dict()
strong = measure_selection(generated, strong_selected, true_task_success).as_dict()
weak, strong

## Result and assertions


In [ ]:
assert weak["oracle@N"] == 1
assert weak["selected_success@N"] == 0
assert weak["selection_gap@N"] == 1
assert weak["selected_id"] == "A"
assert strong["selected_success@N"] == 1
assert strong["selection_gap@N"] == 0
assert strong["selected_id"] == "B"
assert weak["diversity_count"] == 3
print("weak evaluator:", weak)
print("strong evaluator:", strong)

## Stage-01 eligibility still constrains selection

A candidate can score highly and still be ineligible. Selection is downstream of the acceptance boundary, not a way around it.


In [ ]:
with_rejected = CandidateSet.from_iterable(
    [
        Candidate(
            "unsafe",
            "edit_file",
            "perfect repair",
            "highest proxy score",
            accepted_by_stage01=False,
        ),
        Candidate(
            "allowed",
            "edit_file",
            "safe repair",
            "eligible alternative",
            accepted_by_stage01=True,
        ),
    ]
)

selector = CandidateSelector(
    FunctionCandidateEvaluator(
        lambda candidate: 100.0 if candidate.id == "unsafe" else 1.0
    )
)
selected = selector.select_complete(with_rejected)
assert selected.candidate.id == "allowed"
selected

## Pointwise and pairwise selection are different regimes

The shared Stage-02 demo deliberately uses a **pointwise** evaluator: each complete candidate receives a score and the selector chooses among those scores. The current book also distinguishes **pairwise** evaluation, where candidates are compared directly against one another.

That distinction matters because selection quality is a property of the evaluator/selector pair, not of generation alone. This notebook keeps one regime fixed so the causal experiment remains interpretable; it does not treat pointwise and pairwise judging as interchangeable.


## What was earned

**Generation creates opportunity; selection determines whether that opportunity is used.** The implementation can now measure whether failure came from not generating a good option or from choosing badly among available options.

No critique/revision, planning, code mutation, or partial-trajectory search has been introduced.


## Demo API now implemented

```python
from first_principles_agent.candidates import (
    CandidateGenerator, CandidateEvaluator, CandidateSelector, CandidateSet
)
selector.select_complete(candidates, state=None) -> SelectedCandidate
```

Notebook 04 / Chapter 4 consumes a selected candidate and adds **targeted critique, revision, and accept/rollback** as a separate mechanism.
